In [7]:
# ── CELL 1: Imports & Load ────────────────────────────────────
import pandas as pd
import numpy as np
import requests
import time
import pickle
from pathlib import Path
from scipy.stats import mannwhitneyu

ROOT    = Path("..")
MATCH   = ROOT / "data" / "matched"
FIG     = ROOT / "data" / "figures"
EMAIL   = "you@email.com"
HEADERS = {"User-Agent": f"thesis-rq3/1.0 mailto:{EMAIL}"}
BASE    = "https://api.openalex.org"

# Load what we have
juniors  = pd.read_csv(MATCH / "junior_authors_all_conferences.csv")
pairs    = pd.read_csv(MATCH / "matched_pairs_clean.csv")
core     = pd.read_csv(MATCH / "conference_core_ranks.csv")

with open(MATCH / "award_source_map.pkl", "rb") as f:
    award_source_map = pickle.load(f)

# Unique conferences with source IDs
conf_sources = {
    conf: src
    for (conf, year), src in award_source_map.items()
    if src is not None
}
# Keep one source ID per conference (most recent non-null)
conf_source_final = {}
for (conf, year), src in sorted(award_source_map.items()):
    if src:
        conf_source_final[conf] = src

print(f"Conferences with source IDs: {len(conf_source_final)}")
print(list(conf_source_final.keys()))


Conferences with source IDs: 28
['AAAI', 'ACL', 'CHI', 'CIKM', 'CVPR', 'FOCS', 'FSE', 'ICCV', 'ICML', 'ICSE', 'IJCAI', 'INFOCOM', 'KDD', 'MOBICOM', 'NSDI', 'NeurIPS', 'OSDI', 'PLDI', 'PODS', 'S&P', 'SIGIR', 'SIGMOD', 'SODA', 'SOSP', 'STOC', 'UIST', 'VLDB', 'WWW']


In [ ]:
# ── CELL 2: Fetch award paper citations from OpenAlex ─────────
def fetch_work_citations(work_id, retries=3):
    url = f"{BASE}/works/{work_id}"
    params = {"select": "id,cited_by_count,publication_year"}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.json()
        except Exception:
            time.sleep(2 ** attempt)
    return None

# Get unique award work IDs
award_works = juniors[["conference","award_year","work_id"]].drop_duplicates("work_id").reset_index(drop=True)

rows = []
for i, row in award_works.iterrows():
    data = fetch_work_citations(row["work_id"])
    if data:
        rows.append({
            "conference":      row["conference"],
            "award_year":      row["award_year"],
            "work_id":         row["work_id"],
            "cited_by_count":  data.get("cited_by_count", 0),
            "type": "award"
        })
    if i % 50 == 0:
        print(f"  {i}/{len(award_works)} fetched")
    time.sleep(0.3)

award_citations = pd.DataFrame(rows)
award_citations.to_csv(MATCH / "award_paper_citations.csv", index=False)
print(f"\nAward papers fetched: {len(award_citations)}")
print(award_citations["cited_by_count"].describe().round(1))


  0/470 fetched
  50/470 fetched
  250/470 fetched
  300/470 fetched
  400/470 fetched
  500/470 fetched
  550/470 fetched
  600/470 fetched

Award papers fetched: 470
count      470.0
mean       308.2
std       1453.0
min          0.0
25%         45.0
50%        102.5
75%        203.5
max      28027.0
Name: cited_by_count, dtype: float64
